# Programming for Data Science — Final Project
### Master in Data Science — ESILV
**Instructor**: Hugo Alatrista-Salas  

---

### 👥 Team Members & Project Links
* **Maxime LAURENT** (*Team Leader*) — `maxime.laurent@edu.devinci.fr`
* *(Team members to be added)*
* 🔗 **GitHub Repository**: [https://github.com/groupe-dia3-ml/programming-for-data-science](https://github.com/groupe-dia3-ml/programming-for-data-science)

---

# Multidimensional Knowledge Discovery in Databases (KDD) Prototype

Many real-world phenomena can be modeled and analyzed through multidimensional data. A **multidimensional dataset** is defined by three fundamental axes:
1. 🕒 **Temporal Dimension (*When*)**: Timestamps, chronological periods, or duration.
2. 🗺️ **Spatial Dimension (*Where*)**: Latitude/longitude coordinates, countries, or geographic regions.
3. 📊 **Analytical Dimension (*What*)**: Domain characteristics, measurements, impact metrics, and categories.

The goal of this project is to apply the full **KDD process** (data collection, exploration, preprocessing, data mining, and visualization) to extract key indicators that support data-driven decision-making.

## 📑 Table of Contents

1. [Setup & Library Imports](#1-setup--library-imports)
2. [Dataset Context & Domain Meaning](#2-dataset-context--domain-meaning)
3. [Modular Function Definitions](#3-modular-function-definitions)
   - [Data Ingestion & Profiling](#data-ingestion--profiling)
   - [Query 1: Grouping & Aggregation](#query-1-grouping--aggregation)
   - [Query 2: Frequent Pattern Mining](#query-2-frequent-pattern-mining)
   - [Query 3: Temporal Dynamics & Forecasting](#query-3-temporal-dynamics--forecasting)
   - [Query 4: Spatial Information & Clustering](#query-4-spatial-information--clustering)
4. [Indicator Descriptions & Methodological Explanations](#4-indicator-descriptions--methodological-explanations)
5. [Main Execution Block (`if __name__ == '__main__':`)](#5-main-execution-block)
6. [Conclusion & Decision Support Prototype](#6-conclusion--decision-support-prototype)

---
## 1. Setup & Library Imports

We import standard numerical, tabular, geospatial, pattern mining, and interactive visualization libraries:

In [ ]:
# -*- coding: utf-8 -*-
# Import required data science, spatial, temporal, and visualization libraries
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Pattern Mining
from mlxtend.frequent_patterns import apriori, association_rules

# Spatial & Machine Learning
from sklearn.cluster import OPTICS, KMeans

warnings.filterwarnings('ignore')
print("All libraries imported successfully!")

---
## 2. Dataset Context & Domain Meaning

> 📝 **Human-Readable Dataset Description [Task 1 - 2 points]**:
> *(Describe here the chosen multidimensional dataset, its origins, its real-world significance, and how the temporal, spatial, and analytical axes interact).*

### Dimensions of our Multidimensional Dataset:
* **Temporal Axis**: Defines when events occurred (e.g. timestamps, historical dates, seasonal cycles).
* **Spatial Axis**: Defines where events took place (e.g. GPS coordinates `[latitude, longitude]`, administrative areas, countries).
* **Analytical Axis**: Quantitative measurements and qualitative categorizations (e.g. magnitude, financial cost, casualty count, event type).

---
## 3. Modular Function Definitions

*Note: As required by the project instructions, every technique is implemented as an independent function with documented inputs, outputs, and descriptive line comments.*

In [ ]:
# Function for reading and loading a CSV file
# Input: the path to the csv file
# Output: a pandas DataFrame
def load_data(file_path: str) -> pd.DataFrame:
    # Check if the target file exists on disk
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found at: {file_path}")
    
    # Load dataset using pandas
    df = pd.read_csv(file_path)
    
    # Strip whitespace from headers
    df.columns = df.columns.str.strip()
    
    return df

# Function for profiling and summarizing the dataset
# Input: pandas DataFrame
# Output: summary dictionary and prints basic stats
def summarize_dataset(df: pd.DataFrame) -> dict:
    # Compute row and column counts
    n_rows, n_cols = df.shape
    
    # Identify missing values per column
    missing = df.isnull().sum().to_dict()
    
    # Summarize column data types
    dtypes = df.dtypes.astype(str).to_dict()
    
    print(f"Dataset Shape: {n_rows:,} rows, {n_cols} columns")
    print(f"Total Missing Cells: {df.isnull().sum().sum():,}")
    
    return {"rows": n_rows, "cols": n_cols, "missing": missing, "dtypes": dtypes}

In [ ]:
# Query 1: Grouping / Aggregation indicator
# Input: df (DataFrame), group_col (str), target_col (str)
# Output: aggregated DataFrame
def function_query_1_aggregation(df: pd.DataFrame, group_col: str, target_col: str) -> pd.DataFrame:
    # Group by the specified category and calculate sum, mean, max, and count
    agg_res = df.groupby(group_col)[target_col].agg(
        Total_Sum='sum',
        Average='mean',
        Maximum='max',
        Frequency='count'
    ).reset_index()
    
    # Sort descending by Total_Sum
    agg_res = agg_res.sort_values(by='Total_Sum', ascending=False)
    
    return agg_res

In [ ]:
# Query 2: Frequent Pattern Mining indicator
# Input: df (DataFrame), feature_cols (list of str), min_support (float), min_lift (float)
# Output: tuple of (frequent_itemsets, association_rules)
def function_query_2_pattern_mining(df: pd.DataFrame, feature_cols: list, min_support: float = 0.05, min_lift: float = 1.1):
    # Select categorical attributes and drop nulls
    subset = df[feature_cols].dropna().astype(str)
    
    # One-hot encode using pd.get_dummies
    encoded = pd.get_dummies(subset, prefix_sep='=')
    
    # Run Apriori algorithm
    frequent_itemsets = apriori(encoded, min_support=min_support, use_colnames=True)
    
    # Extract association rules
    if not frequent_itemsets.empty:
        rules = association_rules(frequent_itemsets, metric="lift", min_threshold=min_lift)
        rules = rules.sort_values(by="support", ascending=False)
    else:
        rules = pd.DataFrame()
        
    return frequent_itemsets, rules

In [ ]:
# Query 3: Temporal Dynamics & Evolution indicator
# Input: df (DataFrame), date_col (str), value_col (str), freq (str)
# Output: resampled time series DataFrame ready for forecasting
def function_query_3_temporal_trend(df: pd.DataFrame, date_col: str, value_col: str, freq: str = 'M') -> pd.DataFrame:
    # Copy DataFrame to avoid modifying original
    t_df = df.copy()
    
    # Parse datetime column
    t_df['ds'] = pd.to_datetime(t_df[date_col], errors='coerce')
    t_df = t_df.dropna(subset=['ds'])
    
    # Resample time series
    ts = t_df.set_index('ds').resample(freq)[value_col].agg(['sum', 'mean', 'count']).reset_index()
    ts.rename(columns={'sum': 'y'}, inplace=True)
    
    return ts

In [ ]:
# Query 4: Spatial Coordinates & Clustering indicator
# Input: df (DataFrame), lat_col (str), lon_col (str), min_samples (int)
# Output: DataFrame with cluster labels
def function_query_4_spatial_clustering(df: pd.DataFrame, lat_col: str, lon_col: str, min_samples: int = 5) -> pd.DataFrame:
    # Filter valid coordinates
    s_df = df.dropna(subset=[lat_col, lon_col]).copy()
    coords = s_df[[lat_col, lon_col]].values
    
    # Fit OPTICS clustering
    clusterer = OPTICS(min_samples=min_samples)
    s_df['cluster'] = clusterer.fit_predict(coords)
    
    return s_df

---
## 4. Indicator Descriptions & Methodological Explanations

> ⚠️ *Mandatory: Every indicator must have an accompanying Markdown box describing how it is calculated, what it represents, and how it will be interpreted.*

### Indicator 1: Aggregation & Volume Distribution [Query 1 - 1 pt]
* **Calculation:** Computed via `groupby(category)` computing `sum`, `mean`, `max`, and `count`.
* **Interpretation:** Identifies the primary drivers of impact (e.g. which category causes the highest cumulative damages or event frequencies).

### Indicator 2: High-Affinity Association Rules [Query 2 - 2 pts]
* **Calculation:** Extracted with Apriori with minimum support threshold $\sigma$ and filtered by $\text{Lift} \ge 1.1$.
* **Interpretation:** Unveils hidden co-occurrences between categorical characteristics (e.g., severe impact $\rightarrow$ specific region).

### Indicator 3: Temporal Trajectory & Forecasting [Query 3 - 2 pts]
* **Calculation:** Time-series aggregation resampled by month/quarter and modeled with additive trend and seasonality.
* **Interpretation:** Detects seasonal cycles, historical acceleration, and future expected event volume.

### Indicator 4: Spatial Density Clusters [Query 4 - 2 pts]
* **Calculation:** Density-based clustering (OPTICS) applied directly to geographic coordinate pairs $(\text{Latitude}, \text{Longitude})$.
* **Interpretation:** Highlights high-density spatial hotspots and isolates anomalous/dispersed events.

---
## 5. Main Execution Block

Following the template on Page 4 of `Final_project.pdf`, all functions are orchestrated inside the `if __name__ == '__main__':` block:

In [ ]:
# Main block
if __name__ == "__main__":
    print("=" * 60)
    print("STARTING KDD FINAL PROJECT EXECUTION PIPELINE")
    print("=" * 60)
    
    # Step 1: Data collection
    # file_path = "data/my_multidimensional_dataset.csv"
    # data = load_data(file_path)
    # summary = summarize_dataset(data)
    
    # Step 2: Indicators construction
    # res_q1 = function_query_1_aggregation(data, group_col='...', target_col='...')
    # res_q2_items, res_q2_rules = function_query_2_pattern_mining(data, feature_cols=['...', '...'])
    # res_q3 = function_query_3_temporal_trend(data, date_col='...', value_col='...')
    # res_q4 = function_query_4_spatial_clustering(data, lat_col='...', lon_col='...')
    
    # Step 3: Launch visualization prototype
    print("Pipeline ready for execution on validated dataset!")
    print("=" * 60)

---
## 6. Conclusion & Decision Support Prototype

This notebook forms the analytical core of the final project deliverable. The indicators constructed above are seamlessly exported and visualized inside the interactive Plotly Dash dashboard (`src/dashboard.py` / `Laurent_dashboard.html`).

---
> 🚀 *May the Force be with you!*